In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime

# === CONFIGURATION ===
DB_URL = "sqlite:///startup_dwh.db"  # Ganti dengan koneksi database sesungguhnya jika perlu
engine = create_engine(DB_URL)

# === EXTRACT ===
df_company_raw = pd.read_csv("data/company.csv")
df_funding_raw = pd.read_csv("data/funding_rounds.csv")
df_acquisition_raw = pd.read_csv("data/acquisition.csv")
df_ipo_raw = pd.read_csv("data/ipos.csv")
df_funds_raw = pd.read_csv("data/funds.csv")
df_investment_raw = pd.read_csv("data/investment.csv")
df_people_raw = pd.read_csv("data/people.csv")
df_relationship_raw = pd.read_csv("data/relationship.csv")
df_milestone_raw = pd.read_csv("data/milestones.csv")

# === TRANSFORM ===

# dim_company
df_company = df_company_raw.rename(columns={
    "object_id": "company_id",
    "description": "description",
    "city": "city",
    "state_code": "state_code",
    "country_code": "country_code",
    "latitude": "latitude",
    "longitude": "longitude",
    "created_at": "created_at"
})[[
    "company_id", "description", "city", "state_code", "country_code", "latitude", "longitude", "created_at"
]].drop_duplicates()

# dim_people
df_people = df_people_raw.rename(columns={
    "people_id": "person_id",
    "object_id": "company_id",
    "first_name": "first_name",
    "last_name": "last_name",
    "birthplace": "birthplace",
    "affiliation_name": "affiliation_name"
})[[
    "person_id", "first_name", "last_name", "birthplace", "affiliation_name", "company_id"
]].drop_duplicates()

# dim_relationship
df_relationship = df_relationship_raw.rename(columns={
    "relationship_id": "relationship_id",
    "person_object_id": "person_id",
    "relationship_object_id": "company_id",
    "start_at": "start_at",
    "end_at": "end_at",
    "is_past": "is_past",
    "sequence": "sequence",
    "title": "title"
})[[
    "relationship_id", "person_id", "company_id", "start_at", "end_at", "is_past", "sequence", "title"
]].drop_duplicates()

# dim_milestone
df_milestone = df_milestone_raw.rename(columns={
    "milestone_id": "milestone_id",
    "object_id": "company_id",
    "milestone_code": "milestone_code",
    "milestone_at": "milestone_at",
    "description": "description"
})[[
    "milestone_id", "company_id", "milestone_code", "milestone_at", "description"
]].drop_duplicates()

# dim_date (gabungan dari semua tanggal utama)
date_cols = [
    df_funding_raw['funded_at'],
    df_acquisition_raw['acquired_at'],
    df_ipo_raw['public_at'],
    df_funds_raw['funded_at'],
    df_milestone_raw['milestone_at']
]
dates = pd.concat(date_cols).dropna().drop_duplicates()
df_date = pd.DataFrame(dates)
df_date.columns = ['date']
df_date['date_key'] = df_date['date'].apply(lambda x: x.strftime('%Y%m%d')).astype(int)
df_date['year'] = df_date['date'].dt.year
df_date['month'] = df_date['date'].dt.month
df_date['day'] = df_date['date'].dt.day
df_date['quarter'] = df_date['date'].dt.quarter
df_date['weekday'] = df_date['date'].dt.strftime('%A')
df_date['full_date'] = df_date['date']
df_date = df_date.drop(columns='date')

# fact_funding_rounds
df_funding_raw['funded_at'] = pd.to_datetime(df_funding_raw['funded_at'])
df_funding = df_funding_raw.copy()
df_funding['date_key'] = df_funding['funded_at'].dt.strftime('%Y%m%d').astype(int)
df_funding = df_funding[[
    "funding_round_id", "date_key", "object_id", "funding_round_type", "funding_round_code",
    "raised_amount_usd", "pre_money_valuation_usd", "post_money_valuation_usd",
    "is_first_round", "is_last_round"
]].rename(columns={"object_id": "company_id"})

# fact_acquisitions
df_acquisition_raw['acquired_at'] = pd.to_datetime(df_acquisition_raw['acquired_at'])
df_acquisition = df_acquisition_raw.copy()
df_acquisition['date_key'] = df_acquisition['acquired_at'].dt.strftime('%Y%m%d').astype(int)
df_acquisition = df_acquisition[[
    "acquisition_id", "date_key", "acquiring_object_id", "acquired_object_id",
    "term_code", "price_amount"
]].rename(columns={
    "acquiring_object_id": "acquiring_company_id",
    "acquired_object_id": "acquired_company_id"
})

# fact_ipos
df_ipo_raw['public_at'] = pd.to_datetime(df_ipo_raw['public_at'])
df_ipo = df_ipo_raw.copy()
df_ipo['date_key'] = df_ipo['public_at'].dt.strftime('%Y%m%d').astype(int)
df_ipo = df_ipo[[
    "ipo_id", "date_key", "object_id", "valuation_amount", "raised_amount", "stock_symbol"
]].rename(columns={"object_id": "company_id"})

# fact_funds
df_funds_raw['funded_at'] = pd.to_datetime(df_funds_raw['funded_at'])
df_funds = df_funds_raw.copy()
df_funds['date_key'] = df_funds['funded_at'].dt.strftime('%Y%m%d').astype(int)
df_funds = df_funds[[
    "fund_id", "date_key", "object_id", "name", "raised_amount"
]].rename(columns={"object_id": "company_id"})

# fact_investments
df_investment = df_investment_raw[[
    "investment_id", "funding_round_id", "funded_object_id", "investor_object_id"
]]

# === LOAD ===
df_company.to_sql("dim_company", engine, if_exists="append", index=False)
df_people.to_sql("dim_people", engine, if_exists="append", index=False)
df_relationship.to_sql("dim_relationship", engine, if_exists="append", index=False)
df_milestone.to_sql("dim_milestone", engine, if_exists="append", index=False)
df_date.to_sql("dim_date", engine, if_exists="append", index=False)

df_funding.to_sql("fact_funding_rounds", engine, if_exists="append", index=False)
df_acquisition.to_sql("fact_acquisitions", engine, if_exists="append", index=False)
df_ipo.to_sql("fact_ipos", engine, if_exists="append", index=False)
df_funds.to_sql("fact_funds", engine, if_exists="append", index=False)
df_investment.to_sql("fact_investments", engine, if_exists="append", index=False)

print("ETL selesai untuk semua tabel.")
